# 02 骨干网络比较（E1）

比较 ResNet18、MobileNetV2、EfficientNet-B0。固定 Freeze、Basic、数据划分、种子与预算，只改变骨干网络。

**前置条件：** 完成 01 数据准备；三个模型均从 ImageNet 预训练权重初始化。

全部模型选择只依据 **PlantVillage 验证集 Macro-F1**，本 Notebook 不读取 PlantVillage 测试图片或 PlantDoc。训练参数来自配置，默认 224×224、batch size 16、最多 10 轮、AdamW、交叉熵与 Early Stopping；改变配置前先记录理由。

`RUN_TRAINING=False` 时只读取配置和已有记录，不下载权重或训练。打开训练后，已完成且配置和数据指纹一致的候选自动跳过；未完成候选采用新的 run_id 重新训练，旧中断记录仍可识别。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请在项目根目录或 notebooks 目录打开本 Notebook。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, load_labels

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")
CLASS_NAMES = [item["label"] for item in load_labels(PROJECT_ROOT)]
assert len(CLASS_NAMES) == 8 and len(set(CLASS_NAMES)) == 8
assert [item["label_index"] for item in LABELS["classes"]] == list(range(8))
print("项目目录：", PROJECT_ROOT)
print("类别数量：", len(CLASS_NAMES), "；随机种子：", BASE["seed"])


## 1. 查看计划与运行状态

先确认候选和基础配置。MPS 可用时优先使用，必要时回退 CPU。每个正式实验有独立目录，保存实际配置、标签、数据指纹、环境、状态、最佳权重和历史；不得把烟测结果用作模型选择。


In [ ]:
from src.experiments import run_stage, stage_results, run_statuses, select_best, selected_run

STAGE = "E1"
RUN_TRAINING = False
SAVE_SELECTION = False

print("阶段：", STAGE)
print("候选：", EXPERIMENTS.get("baseline_models"))
print("基础训练参数：")
display(BASE["training"])
print("策略学习率：", BASE["training"].get("lr_by_strategy"))
print("已选择骨干：", EXPERIMENTS.get("selected_backbone"))
print("已选择微调：", EXPERIMENTS.get("selected_finetuning"))
print("已选择增强：", EXPERIMENTS.get("selected_augmentation"))
display(pd.DataFrame(run_statuses(PROJECT_ROOT)))


## 2. 运行本阶段训练

确认配置后，在上一格把 `RUN_TRAINING` 改为 `True`，重新运行上一格，再运行本格。训练过程可能持续较久；完成后查看下一节真实结果。训练异常会保留失败状态；内核被强制关闭时，状态和不完整文件仍可识别，重新运行会建立新实验。


In [ ]:
if not RUN_TRAINING:
    print("训练关闭：目前只展示计划与已有结果。")
else:
    for run_dir in run_stage(PROJECT_ROOT, STAGE):
        print("已完成：", run_dir.name)


## 3. 汇总完成的实验与训练曲线

仅汇总当前八类、当前清单和当前阶段参数一致的已完成正式实验。`val_macro_f1` 是最佳验证轮次对应指标，训练耗时按实际记录读取。尚无结果时不生成占位指标；中断训练不参与比较。


In [ ]:
try:
    records = stage_results(PROJECT_ROOT, STAGE)
except (ValueError, FileNotFoundError) as error:
    records = []
    print("尚未满足数据准备或前置阶段条件：", error)
if not records:
    print("尚无本阶段完成的八分类正式结果。")
else:
    table = pd.DataFrame(records).sort_values("val_macro_f1", ascending=False)
    display(table)
    table_dir = PROJECT_ROOT / "outputs" / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(table_dir / f"{STAGE.lower()}_validation.csv", index=False, encoding="utf-8-sig")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for record in records:
        history = pd.read_csv(PROJECT_ROOT / "outputs" / "runs" / record["run_id"] / "history.csv")
        axes[0].plot(history["epoch"], history["val_macro_f1"], label=record["run_id"])
        axes[1].plot(history["epoch"], history["val_loss"], label=record["run_id"])
    axes[0].set(xlabel="Epoch", ylabel="Validation Macro-F1")
    axes[1].set(xlabel="Epoch", ylabel="Validation loss")
    for axis in axes:
        axis.legend(fontsize=6)
    plt.tight_layout()
    figure_dir = PROJECT_ROOT / "outputs" / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / f"{STAGE.lower()}_validation.png", dpi=160, bbox_inches="tight")
    plt.show()


## 4. 保存骨干网络选择

查看结果后，将第 1 节 `SAVE_SELECTION` 改为 `True`，重新运行该格，再运行本格。函数要求当前阶段的全部候选都已完成，再按验证集 Macro-F1 选择并写回 `configs/experiments.yaml` 的 `selected_*` 和 `selected_runs`。并列时使用固定规则，详细比较仍保留在表格与记录中。

选择后切换到下一本 Notebook；E3 选定实验就是 Notebook 05 最终评价和 06—07 展示的模型。不要根据测试集结果回头改选。


In [ ]:
if not SAVE_SELECTION:
    print("尚未保存新选择；查看结果后将 SAVE_SELECTION 设为 True。")
else:
    try:
        selected = select_best(PROJECT_ROOT, STAGE)
        print("已按验证集结果保存选择：")
        display(selected)
    except (ValueError, FileNotFoundError) as error:
        print("尚不能选择：", error)


## 结论记录（运行后填写）

- 数据清单、设备、随机种子、训练预算与实际学习率：待填写。
- 各候选验证集 Macro-F1、Accuracy、耗时及训练曲线：待填写。
- 选定骨干网络与依据：待填写。
- 部分微调的实际解冻范围、过拟合现象或增强影响：待结合本阶段记录填写。
- 当前默认仅种子 42；单次结果不能反映随机波动，也不能证明统计显著性。
